In [0]:
# =====================================================
# 1️⃣ Widgets
# =====================================================
# REQUIRED PARAMETERS (jobs inject them; manual runs must supply them):
#   run_id           : numeric run identifier from the orchestrating job
#   table_metadata   : JSON string for the table row (from <catalog>.metadata.tables)
#   table_parameters : JSON string (load_type, primary_key, watermark_column)
#   catalog_name     : Unity Catalog name (e.g. 'banking') — see config/*.json
#   neon_host / neon_port / neon_database : non-secret Neon connection settings — see config/*.json
#   volume_base      : source volume root for blob loads (e.g. /Volumes/banking/source/volume)

import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

dbutils.widgets.text(
    "table_metadata",
    "{'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', "
    "'source_schema': 'banking', 'source_table': 'customers', "
    "'source_path': '', 'bronze_schema': 'bronze', "
    "'silver_schema': 'silver', 'active_flag': 'True', "
    "'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}"
)

dbutils.widgets.text(
    "table_parameters",
    "{'load_type': 'MERGE', "
    "'primary_key': 'customer_id', "
    "'watermark_column': 'updated_at'}"
)

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("neon_host", "")
dbutils.widgets.text("neon_port", "")
dbutils.widgets.text("neon_database", "")
dbutils.widgets.text("volume_base", "")

run_id = dbutils.widgets.get("run_id").strip()

# ---- Fail-fast validation ----
if not run_id:
    raise ValueError(
        "run_id is required. Jobs inject it (master job passes {{job.run_id}}); "
        "manual runs must supply a unique numeric value."
    )
if not run_id.isdigit():
    raise ValueError(
        f"run_id must be numeric (pipeline_runs.run_id is BIGINT). Got: {run_id!r}"
    )

# Parse JSON safely
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_parameters = json.loads(dbutils.widgets.get("table_parameters").replace("'", '"'))

print("Table Metadata:", table_metadata)
print("Table Parameters:", table_parameters)
print(f"Run ID: {run_id}")

In [0]:
# =====================================================
# 2️⃣ Extract Variables
# =====================================================

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"].lower()
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]

load_type = table_parameters.get("load_type")
watermark_column = table_parameters.get("watermark_column")

catalog_name = dbutils.widgets.get("catalog_name").strip()
if not catalog_name:
    raise ValueError("catalog_name is required (e.g. 'banking' — see config/*.json)")

bronze_table_fqn = f"{catalog_name}.{bronze_schema}.{table_name}"

print(f"Target Bronze Table: {bronze_table_fqn}")

In [0]:
# =====================================================
# Make and entry to audit table
# =====================================================
entry_exists = spark.sql(f"""
    SELECT 1
    FROM {catalog_name}.metadata.pipeline_runs
    WHERE run_id = {run_id} AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE {catalog_name}.metadata.pipeline_runs
        SET
            layer = 'Silver',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id} AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO {catalog_name}.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Silver',
            TIMESTAMP('{start_time}'),
            NULL,  -- end time
            'INPROGRESS',
            NULL, --number of records
            NULL -- error message
        )
    """)

In [0]:
# =====================================================
# 3️⃣ Get Last Watermark (For Filtering Only)
# =====================================================

last_watermark = None

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM {catalog_name}.metadata.table_watermarks
        WHERE table_id = {table_id}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print("Last Watermark:", last_watermark)

In [0]:
%sql
create schema if not exists ${catalog_name}.bronze

In [0]:

# =====================================================
# 4️⃣ Read Source
# =====================================================
# NOTE: source_system label 'sqlserver' is a LEGACY NAME in the metadata registry.
#       The actual JDBC source is Neon PostgreSQL. Renaming the label is PENDING
#       (must change atomically in metadata.tables + master job + this branch).

try:
    if source_system == "sqlserver":
    
        # ---------------------------------------------
        # Read Neon credentials from Databricks Secrets
        # ---------------------------------------------

        neon_username = dbutils.secrets.get(
            scope="banking-scope",
            key="neon-username"
        )

        neon_password = dbutils.secrets.get(
            scope="banking-scope",
            key="neon-password"
        )

        # ---------------------------------------------
        # PostgreSQL JDBC Configuration (non-secret settings
        # from widgets — see config/*.json)
        # ---------------------------------------------

        neon_host = dbutils.widgets.get("neon_host").strip()
        neon_port = dbutils.widgets.get("neon_port").strip()
        neon_database = dbutils.widgets.get("neon_database").strip()

        missing = [k for k, v in {
            "neon_host": neon_host,
            "neon_port": neon_port,
            "neon_database": neon_database
        }.items() if not v]
        if missing:
            raise ValueError(
                f"Missing required connection widgets: {missing}. "
                "Supply values from config/*.json."
            )

        jdbc_url = (
            f"jdbc:postgresql://{neon_host}:"
            f"{neon_port}/{neon_database}?sslmode=require"
        )

        jdbc_properties = {
            "user": neon_username,
            "password": neon_password,
            "driver": "org.postgresql.Driver"
        }

        # ---------------------------------------------
        # Build Incremental / Full Load Query
        # ---------------------------------------------

        if load_type in ["APPEND", "MERGE"] and last_watermark:

            query = f"""
                (
                    SELECT *
                    FROM {source_schema}.{source_table}
                    WHERE {watermark_column} > '{last_watermark}'
                ) AS src
            """

        else:

            query = f"""
                (
                    SELECT *
                    FROM {source_schema}.{source_table}
                ) AS src
            """

        # ---------------------------------------------
        # Read Data from Neon PostgreSQL
        # ---------------------------------------------

        source_df = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=jdbc_properties
        )

    elif source_system == "blob":

        volume_base = dbutils.widgets.get("volume_base").strip().rstrip("/")
        if not volume_base:
            raise ValueError(
                "volume_base is required for blob sources "
                "(e.g. /Volumes/banking/source/volume — see config/*.json)"
            )

        source_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option(
                "cloudFiles.schemaLocation",
                f"{volume_base}/_schema/{table_name}"
            )
            .option("header", "true")
            .load(source_path)
        )

    else:
        raise ValueError(
            f"Unsupported source_system: {source_system}"
        )

    # =====================================================
    # 5️⃣ Add insert_timestamp
    # =====================================================

    source_df = source_df.withColumn(
        "insert_timestamp",
        current_timestamp()
    )

    # =====================================================
    # 6️⃣ Write to Bronze (Append Only)
    # =====================================================

    if source_system == "blob":

        (
            source_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"{volume_base}/_checkpoints/{table_name}"
            )
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )

        records_read = None

    else:

        (
            source_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(bronze_table_fqn)
        )

        records_read = source_df.count()

    print("Source → Bronze Load Completed Successfully.")
    print("Watermark will be updated after Silver load.")

except Exception as e:

    end_time = spark.sql(
        "SELECT current_timestamp()"
    ).collect()[0][0]

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        UPDATE {catalog_name}.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = {'NULL' if not error_message else "'" + error_message + "'"}
        WHERE table_id = {table_id}
          AND run_id = {run_id}
    """)

    raise
